# ETL — ordenes_mantenimiento — Mantenimiento preventivo de bomba centrífuga

**Fuente:** snapshot `data/raw/ordenes_mantenimiento_raw.csv` (extraído de PostgreSQL).

Este notebook procesa únicamente la tabla `ordenes_mantenimiento`: carga el CSV crudo, aplica la transformación y limpieza y exporta el dataframe limpio a `data/processed/`.

## 1. Configuración

Imports, rutas del proyecto y carga de variables de entorno.

In [14]:
from pathlib import Path
import os
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# --- Rutas del proyecto ---
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / ".env").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_CATALOGO_FALLAS = PROJECT_ROOT / "data" / "raw" / "catalogo_fallas_raw.csv"
RAW_DATA_EQUIPOS = PROJECT_ROOT / "data" / "raw" / "equipos_raw.csv"
RAW_DATA_LECTURAS_SENSORES = PROJECT_ROOT / "data" / "raw" / "lecturas_sensores_raw.csv"
RAW_DATA_ORDENES_MANTENIMIENTO = PROJECT_ROOT / "data" / "raw" / "ordenes_mantenimiento_raw.csv"

PROCESSED_FILE = PROCESSED_DIR / "ordenes_mantenimiento_clean.parquet"



## 2. Análisis, transformación y limpieza

### Tabla ordenes_mantenimiento_raw

In [15]:
df_ordenes_mantenimiento = pd.read_csv(RAW_DATA_ORDENES_MANTENIMIENTO)
# df_a (data frame analisis - data set de analisis) copia del dataset para mantener una copia original intacta (df)
df_ordenes_mantenimiento_w = df_ordenes_mantenimiento.copy()

df_ordenes_mantenimiento_w.head()

,orden_id,equipo_id,tipo_mantenimiento,fecha_inicio,fecha_fin,costo_usd,codigo_falla
0,1,1,Correctivo,2026-10-31 22:00:00,2026-10-31 23:30:00,3500.0,F01


In [16]:
df_ordenes_mantenimiento_w.shape

(1, 7)

In [17]:
df_ordenes_mantenimiento_w.info()

<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   orden_id            1 non-null      int64  
 1   equipo_id           1 non-null      int64  
 2   tipo_mantenimiento  1 non-null      str    
 3   fecha_inicio        1 non-null      str    
 4   fecha_fin           1 non-null      str    
 5   costo_usd           1 non-null      float64
 6   codigo_falla        1 non-null      str    
dtypes: float64(1), int64(2), str(4)
memory usage: 243.0 bytes


In [18]:
print('nulos: ', df_ordenes_mantenimiento_w.isna().sum().sum())
print('duplicados: ', df_ordenes_mantenimiento_w.duplicated().sum())

nulos:  0
duplicados:  0


In [19]:
df_ordenes_mantenimiento_w.columns

Index(['orden_id', 'equipo_id', 'tipo_mantenimiento', 'fecha_inicio',
       'fecha_fin', 'costo_usd', 'codigo_falla'],
      dtype='str')

In [20]:
print(df_ordenes_mantenimiento_w['tipo_mantenimiento'].unique())

<ArrowStringArray>
['  Correctivo  ']
Length: 1, dtype: str


In [21]:
df_ordenes_mantenimiento_w['tipo_mantenimiento'] = df_ordenes_mantenimiento_w['tipo_mantenimiento'].str.strip()
print(df_ordenes_mantenimiento_w['tipo_mantenimiento'].unique())

<ArrowStringArray>
['Correctivo']
Length: 1, dtype: str


In [22]:
df_ordenes_mantenimiento_w['tipo_mantenimiento'] = df_ordenes_mantenimiento_w['tipo_mantenimiento'].replace({
    'Correctivo' : 'correctivo'
})
print(df_ordenes_mantenimiento_w['tipo_mantenimiento'].unique())

<ArrowStringArray>
['correctivo']
Length: 1, dtype: str


In [23]:
df_ordenes_mantenimiento_w['fecha_inicio'] = pd.to_datetime(
    df_ordenes_mantenimiento_w['fecha_inicio'],
    format='%Y-%m-%d %H:%M:%S',
    errors='coerce'
)

df_ordenes_mantenimiento_w['fecha_fin'] = pd.to_datetime(
    df_ordenes_mantenimiento_w['fecha_fin'],
    format='%Y-%m-%d %H:%M:%S',
    errors='coerce'
)

In [24]:
print('Resumen del Dataset Work luego de transformaciones y limpieza')
print('=' * 61)
print('DF_Analysis: ', df_ordenes_mantenimiento.shape, "-> DF_Work: ", df_ordenes_mantenimiento_w.shape)
print('=' * 61)
print('Columnas antes: ', list(df_ordenes_mantenimiento.columns))
print('Columnas después:', list(df_ordenes_mantenimiento_w.columns))
print('=' * 61)
print('DF_Work tiene: ', df_ordenes_mantenimiento_w.isna().sum().sum(), ' Valores Nulos')
print('DF_Work tiene: ', df_ordenes_mantenimiento_w.duplicated().sum(), ' Valores Duplicados')
print('=' * 61)
print('Tipos de datos finales:')
print(df_ordenes_mantenimiento_w.dtypes.to_string())
print('=' * 61)

Resumen del Dataset Work luego de transformaciones y limpieza
DF_Analysis:  (1, 7) -> DF_Work:  (1, 7)
Columnas antes:  ['orden_id', 'equipo_id', 'tipo_mantenimiento', 'fecha_inicio', 'fecha_fin', 'costo_usd', 'codigo_falla']
Columnas después: ['orden_id', 'equipo_id', 'tipo_mantenimiento', 'fecha_inicio', 'fecha_fin', 'costo_usd', 'codigo_falla']
DF_Work tiene:  0  Valores Nulos
DF_Work tiene:  0  Valores Duplicados
Tipos de datos finales:
orden_id                       int64
equipo_id                      int64
tipo_mantenimiento               str
fecha_inicio          datetime64[us]
fecha_fin             datetime64[us]
costo_usd                    float64
codigo_falla                     str


In [25]:
df_ordenes_mantenimiento_w.head()

,orden_id,equipo_id,tipo_mantenimiento,fecha_inicio,fecha_fin,costo_usd,codigo_falla
0,1,1,correctivo,2026-10-31 22:00:00,2026-10-31 23:30:00,3500.0,F01


## 3. Exportación a Parquet

Se guarda el dataframe limpio `df_ordenes_mantenimiento_w` en `data/processed/ordenes_mantenimiento_clean.parquet`.

In [26]:
# Exportación del dataframe limpio a Parquet
archivo_temporal = PROCESSED_FILE.with_suffix(".parquet.tmp")

df_ordenes_mantenimiento_w.to_parquet(archivo_temporal, index=False)
archivo_temporal.replace(PROCESSED_FILE)

print(f"Filas: {len(df_ordenes_mantenimiento_w):,} | Columnas: {df_ordenes_mantenimiento_w.shape[1]}")

Filas: 1 | Columnas: 7
